# MediShelf - Data Cleaning

## Notebook 01: Data Cleaning and Preparation

This notebook inspects, validates and prepares the synthetic blood-demand dataset and daily pharmaceutical-sales dataset for exploratory data analysis and forecasting.

**Workflow:** inspect raw data → validate data types and consistency → assess missing values, duplicates and potential outliers → prepare clean datasets → export processed CSV files.

**Important:** The blood-demand dataset is synthetic. Its suitability for real-world hospital forecasting must therefore be interpreted cautiously.

## 1. Setup and file paths

In [19]:
import os
import numpy as np
import pandas as pd

RAW_BLOOD_PATH = "../data/raw/synthetic_blood_demand_data.csv"
RAW_PHARMA_PATH = "../data/raw/salesdaily.csv"
PROCESSED_DIR = "../data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Blood raw file exists:", os.path.exists(RAW_BLOOD_PATH))
print("Pharmaceutical raw file exists:", os.path.exists(RAW_PHARMA_PATH))
print("Processed directory:", os.path.abspath(PROCESSED_DIR))

Blood raw file exists: True
Pharmaceutical raw file exists: True
Processed directory: C:\Users\Administrator\MediShelf\data\processed


## 2. Blood-demand dataset

### 2.1 Load and inspect the raw data

In [20]:
blood_raw = pd.read_csv(RAW_BLOOD_PATH)

print("Shape:", blood_raw.shape)
display(blood_raw.head())
display(blood_raw.tail())
print("Columns:", blood_raw.columns.tolist())
blood_raw.info()

Shape: (1827, 10)


,Date,DayOfWeek,Month,Population,Events,HistoricalBloodUsage,HospitalAdmissions,BloodDonorsAvailable,Temperature,PredictedBloodDemand
0,2020-01-01,2,1,171958,0,174,98,34,24.799194,258
1,2020-01-02,3,1,196867,0,187,71,55,21.590136,197
2,2020-01-03,4,1,181932,0,132,82,82,38.866004,182
3,2020-01-04,5,1,153694,0,74,88,88,30.089531,60
4,2020-01-05,6,1,169879,0,117,68,93,38.348783,117


,Date,DayOfWeek,Month,Population,Events,HistoricalBloodUsage,HospitalAdmissions,BloodDonorsAvailable,Temperature,PredictedBloodDemand
1822,2024-12-27,4,12,162356,0,222,76,26,25.105501,302
1823,2024-12-28,5,12,160944,0,199,73,65,26.237069,192
1824,2024-12-29,6,12,195282,0,209,38,105,39.298114,150
1825,2024-12-30,0,12,66456,0,246,79,40,36.256685,302
1826,2024-12-31,1,12,81367,0,264,89,20,32.659219,367


Columns: ['Date', 'DayOfWeek', 'Month', 'Population', 'Events', 'HistoricalBloodUsage', 'HospitalAdmissions', 'BloodDonorsAvailable', 'Temperature', 'PredictedBloodDemand']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1827 entries, 0 to 1826
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  1827 non-null   object 
 1   DayOfWeek             1827 non-null   int64  
 2   Month                 1827 non-null   int64  
 3   Population            1827 non-null   int64  
 4   Events                1827 non-null   int64  
 5   HistoricalBloodUsage  1827 non-null   int64  
 6   HospitalAdmissions    1827 non-null   int64  
 7   BloodDonorsAvailable  1827 non-null   int64  
 8   Temperature           1827 non-null   float64
 9   PredictedBloodDemand  1827 non-null   int64  
dtypes: float64(1), int64(8), object(1)
memory usage: 142.9+ KB


In [21]:
display(blood_raw.describe(include="all").T)
print("Missing values:")
display(blood_raw.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", blood_raw.duplicated().sum())

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Date,1827,1827,2020-01-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DayOfWeek,1827.0,NaN,NaN,NaN,3.0,2.000548,0.0,1.0,3.0,5.0,6.0
Month,1827.0,NaN,NaN,NaN,6.521073,3.450157,1.0,4.0,7.0,10.0,12.0
Population,1827.0,NaN,NaN,NaN,125902.013136,43531.328946,50126.0,87792.5,126323.0,164177.5,199859.0
Events,1827.0,NaN,NaN,NaN,0.047619,0.213017,0.0,0.0,0.0,0.0,1.0
HistoricalBloodUsage,1827.0,NaN,NaN,NaN,177.652436,71.575611,50.0,117.0,180.0,238.0,299.0
HospitalAdmissions,1827.0,NaN,NaN,NaN,64.894362,20.301757,30.0,47.0,65.0,82.0,99.0
BloodDonorsAvailable,1827.0,NaN,NaN,NaN,84.253421,37.026689,20.0,52.0,85.0,115.0,149.0
Temperature,1827.0,NaN,NaN,NaN,27.414917,7.224292,15.033888,21.246625,27.409973,33.688241,39.993532
PredictedBloodDemand,1827.0,NaN,NaN,NaN,183.963875,80.178284,50.0,119.0,184.0,244.5,427.0


Missing values:


,missing_count
Date,0
DayOfWeek,0
Month,0
Population,0
Events,0
HistoricalBloodUsage,0
HospitalAdmissions,0
BloodDonorsAvailable,0
Temperature,0
PredictedBloodDemand,0


Exact duplicate rows: 0


### 2.2 Convert and validate dates

In [22]:
blood = blood_raw.copy()
blood["Date"] = pd.to_datetime(blood["Date"], errors="coerce")

print("Invalid/unparseable dates:", blood["Date"].isna().sum())
print("Duplicate dates:", blood["Date"].duplicated().sum())
print("Chronological before sorting:", blood["Date"].is_monotonic_increasing)
print("Date range:", blood["Date"].min(), "to", blood["Date"].max())

Invalid/unparseable dates: 0
Duplicate dates: 0
Chronological before sorting: True
Date range: 2020-01-01 00:00:00 to 2024-12-31 00:00:00


In [23]:
# Sort by date so the cleaned dataset is chronologically ordered.
blood = blood.sort_values("Date").reset_index(drop=True)

print("Chronological after sorting:", blood["Date"].is_monotonic_increasing)
print("Unique dates:", blood["Date"].nunique())
print("Rows:", len(blood))

Chronological after sorting: True
Unique dates: 1827
Rows: 1827


### 2.3 Validate calendar fields

In [24]:
# Compare stored calendar fields with values derived from Date.
month_mismatch = (blood["Date"].dt.month != blood["Month"]).sum()
weekday_mismatch = (blood["Date"].dt.dayofweek != blood["DayOfWeek"]).sum()

print("Month mismatches:", month_mismatch)
print("DayOfWeek mismatches (Monday=0, Sunday=6):", weekday_mismatch)
print("DayOfWeek values in source:", sorted(blood["DayOfWeek"].dropna().unique()))
print("DayOfWeek values derived from Date:", sorted(blood["Date"].dt.dayofweek.dropna().unique()))

Month mismatches: 0
DayOfWeek mismatches (Monday=0, Sunday=6): 0
DayOfWeek values in source: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
DayOfWeek values derived from Date: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]


### 2.4 Check numeric values and potential outliers

The `Events` column is a binary indicator, so the IQR outlier method is **not** applied to it. Potential outliers in continuous/numeric measures are flagged for investigation only; they are not automatically removed or capped. Extreme values may represent genuine variation in the synthetic data.

In [25]:
numeric_columns = [
    "Population",
    "HistoricalBloodUsage",
    "HospitalAdmissions",
    "BloodDonorsAvailable",
    "Temperature",
    "PredictedBloodDemand"
]
if "Events" in blood.columns:
    print("Events values and counts:")
    display(blood["Events"].value_counts(dropna=False).sort_index().to_frame("count"))

print("Negative-value counts:")
display(blood[numeric_columns].lt(0).sum().to_frame("negative_count"))

outlier_summary = []
for column in numeric_columns:
    values = blood[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((values < lower) | (values > upper)).sum()
    outlier_summary.append({
        "column": column,
        "lower_bound": lower,
        "upper_bound": upper,
        "potential_outliers": int(count)
    })

display(pd.DataFrame(outlier_summary).set_index("column").round(2))

Events values and counts:


,count
Events,
0,1740
1,87


Negative-value counts:


,negative_count
Population,0
HistoricalBloodUsage,0
HospitalAdmissions,0
BloodDonorsAvailable,0
Temperature,0
PredictedBloodDemand,0


,lower_bound,upper_bound,potential_outliers
column,,,
Population,-26785.00,278755.00,0
HistoricalBloodUsage,-64.50,419.50,0
HospitalAdmissions,-5.50,134.50,0
BloodDonorsAvailable,-42.50,209.50,0
Temperature,2.58,52.35,0
PredictedBloodDemand,-69.25,432.75,0


### 2.5 Prepare and validate the cleaned blood dataset

In [26]:
blood_clean = blood.copy()

# Retain the project's relevant analysis fields, in a consistent order.
blood_columns = [
    "Date", "DayOfWeek", "Month", "Population", "Events",
    "HistoricalBloodUsage", "HospitalAdmissions",
    "BloodDonorsAvailable", "Temperature", "PredictedBloodDemand"
]
blood_clean = blood_clean[blood_columns].copy()

print("Cleaned shape:", blood_clean.shape)
print("Missing values:")
display(blood_clean.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", blood_clean.duplicated().sum())
print("Duplicate dates:", blood_clean["Date"].duplicated().sum())
print("Chronological:", blood_clean["Date"].is_monotonic_increasing)
print("Date dtype:", blood_clean["Date"].dtype)

Cleaned shape: (1827, 10)
Missing values:


,missing_count
Date,0
DayOfWeek,0
Month,0
Population,0
Events,0
HistoricalBloodUsage,0
HospitalAdmissions,0
BloodDonorsAvailable,0
Temperature,0
PredictedBloodDemand,0


Exact duplicate rows: 0
Duplicate dates: 0
Chronological: True
Date dtype: datetime64[ns]


In [27]:
blood_output = os.path.join(PROCESSED_DIR, "blood_demand_clean.csv")
blood_clean.to_csv(blood_output, index=False)

print("Saved:", os.path.abspath(blood_output))
print("File created:", os.path.exists(blood_output))

Saved: C:\Users\Administrator\MediShelf\data\processed\blood_demand_clean.csv
File created: True


## 3. Pharmaceutical-sales dataset

### 3.1 Load and inspect the raw data

In [28]:
pharma_raw = pd.read_csv(RAW_PHARMA_PATH)

print("Shape:", pharma_raw.shape)
display(pharma_raw.head())
display(pharma_raw.tail())
print("Columns:", pharma_raw.columns.tolist())
pharma_raw.info()

Shape: (2106, 13)


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,1/2/2014,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday
1,1/3/2014,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday
2,1/4/2014,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday
3,1/5/2014,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0,2014,1,276,Sunday
4,1/6/2014,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0,2014,1,276,Monday


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
2101,10/4/2019,7.34,5.683,2.25,22.45,13.0,0.0,1.0,1.00,2019,10,276,Friday
2102,10/5/2019,3.84,5.010,6.00,25.40,7.0,0.0,0.0,0.33,2019,10,276,Saturday
2103,10/6/2019,4.00,11.690,2.00,34.60,6.0,0.0,5.0,4.20,2019,10,276,Sunday
2104,10/7/2019,7.34,4.507,3.00,50.80,6.0,0.0,10.0,1.00,2019,10,276,Monday
2105,10/8/2019,0.33,1.730,0.50,44.30,20.0,2.0,2.0,0.00,2019,10,190,Tuesday


Columns: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Hour', 'Weekday Name']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2106 entries, 0 to 2105
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   datum         2106 non-null   object 
 1   M01AB         2106 non-null   float64
 2   M01AE         2106 non-null   float64
 3   N02BA         2106 non-null   float64
 4   N02BE         2106 non-null   float64
 5   N05B          2106 non-null   float64
 6   N05C          2106 non-null   float64
 7   R03           2106 non-null   float64
 8   R06           2106 non-null   float64
 9   Year          2106 non-null   int64  
 10  Month         2106 non-null   int64  
 11  Hour          2106 non-null   int64  
 12  Weekday Name  2106 non-null   object 
dtypes: float64(8), int64(3), object(2)
memory usage: 214.0+ KB


In [29]:
display(pharma_raw.describe(include="all").T)
print("Missing values:")
display(pharma_raw.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", pharma_raw.duplicated().sum())

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
datum,2106,2106,1/2/2014,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
M01AB,2106.0,NaN,NaN,NaN,5.033683,2.737579,0.0,3.0,4.99,6.67,17.34
M01AE,2106.0,NaN,NaN,NaN,3.89583,2.133337,0.0,2.34,3.67,5.138,14.463
N02BA,2106.0,NaN,NaN,NaN,3.880441,2.38401,0.0,2.0,3.5,5.2,16.0
N02BE,2106.0,NaN,NaN,NaN,29.917095,15.590966,0.0,19.0,26.9,38.3,161.0
N05B,2106.0,NaN,NaN,NaN,8.853627,5.605605,0.0,5.0,8.0,12.0,54.833333
N05C,2106.0,NaN,NaN,NaN,0.593522,1.092988,0.0,0.0,0.0,1.0,9.0
R03,2106.0,NaN,NaN,NaN,5.512262,6.428736,0.0,1.0,4.0,8.0,45.0
R06,2106.0,NaN,NaN,NaN,2.900198,2.415816,0.0,1.0,2.0,4.0,15.0
Year,2106.0,NaN,NaN,NaN,2016.401235,1.66506,2014.0,2015.0,2016.0,2018.0,2019.0


Missing values:


,missing_count
datum,0
M01AB,0
M01AE,0
N02BA,0
N02BE,0
N05B,0
N05C,0
R03,0
R06,0
Year,0


Exact duplicate rows: 0


### 3.2 Define the pharmaceutical sales variables and validate dates

In [30]:
sales_columns = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

pharma = pharma_raw.copy()
pharma["datum"] = pd.to_datetime(pharma["datum"], errors="coerce")

missing_sales_columns = [c for c in sales_columns if c not in pharma.columns]
if missing_sales_columns:
    raise ValueError(f"Expected sales columns are missing: {missing_sales_columns}")

print("Invalid/unparseable dates:", pharma["datum"].isna().sum())
print("Duplicate dates:", pharma["datum"].duplicated().sum())
print("Date range:", pharma["datum"].min(), "to", pharma["datum"].max())

Invalid/unparseable dates: 0
Duplicate dates: 0
Date range: 2014-01-02 00:00:00 to 2019-10-08 00:00:00


In [31]:
# Validate the source's calendar fields against the parsed date.
year_mismatch = (pharma["datum"].dt.year != pharma["Year"]).sum()
month_mismatch = (pharma["datum"].dt.month != pharma["Month"]).sum()
weekday_mismatch = (
    pharma["datum"].dt.day_name() != pharma["Weekday Name"].astype(str)
).sum()

print("Year mismatches:", year_mismatch)
print("Month mismatches:", month_mismatch)
print("Weekday mismatches:", weekday_mismatch)

Year mismatches: 0
Month mismatches: 0
Weekday mismatches: 0


### 3.3 Assess sales values

In [32]:
print("Sales summary statistics:")
display(pharma[sales_columns].describe().T)

print("Missing sales values:")
display(pharma[sales_columns].isna().sum().to_frame("missing_count"))

print("Negative sales values:")
display(pharma[sales_columns].lt(0).sum().to_frame("negative_count"))

print("Zero sales values:")
display((pharma[sales_columns] == 0).sum().to_frame("zero_count"))

print("Percentage of zero values:")
display((pharma[sales_columns].eq(0).mean().mul(100)).round(2).to_frame("zero_percent"))

Sales summary statistics:


,count,mean,std,min,25%,50%,75%,max
M01AB,2106.0,5.033683,2.737579,0.0,3.00,4.99,6.670,17.340000
M01AE,2106.0,3.895830,2.133337,0.0,2.34,3.67,5.138,14.463000
N02BA,2106.0,3.880441,2.384010,0.0,2.00,3.50,5.200,16.000000
N02BE,2106.0,29.917095,15.590966,0.0,19.00,26.90,38.300,161.000000
N05B,2106.0,8.853627,5.605605,0.0,5.00,8.00,12.000,54.833333
N05C,2106.0,0.593522,1.092988,0.0,0.00,0.00,1.000,9.000000
R03,2106.0,5.512262,6.428736,0.0,1.00,4.00,8.000,45.000000
R06,2106.0,2.900198,2.415816,0.0,1.00,2.00,4.000,15.000000


Missing sales values:


,missing_count
M01AB,0
M01AE,0
N02BA,0
N02BE,0
N05B,0
N05C,0
R03,0
R06,0


Negative sales values:


,negative_count
M01AB,0
M01AE,0
N02BA,0
N02BE,0
N05B,0
N05C,0
R03,0
R06,0


Zero sales values:


,zero_count
M01AB,40
M01AE,36
N02BA,78
N02BE,26
N05B,43
N05C,1430
R03,484
R06,256


Percentage of zero values:


,zero_percent
M01AB,1.90
M01AE,1.71
N02BA,3.70
N02BE,1.23
N05B,2.04
N05C,67.90
R03,22.98
R06,12.16


### 3.4 Check potential sales outliers

Pharmaceutical sales can be intermittent, with many zero-demand days and occasional high-demand days. IQR results are therefore treated as flags for investigation, not proof of errors. Potential outliers are retained unless there is evidence that they are invalid.

In [33]:
outlier_rows = []
for column in sales_columns:
    values = pharma[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((values < lower) | (values > upper)).sum()
    outlier_rows.append({
        "sales_group": column,
        "lower_bound": lower,
        "upper_bound": upper,
        "potential_outliers": int(count)
    })

display(pd.DataFrame(outlier_rows).set_index("sales_group").round(2))

,lower_bound,upper_bound,potential_outliers
sales_group,,,
M01AB,-2.50,12.18,26
M01AE,-1.86,9.34,40
N02BA,-2.80,10.00,31
N02BE,-9.95,67.25,48
N05B,-5.50,22.50,48
N05C,-1.50,2.50,160
R03,-9.50,18.50,124
R06,-3.50,8.50,81


### 3.5 Check the daily date sequence

The source includes an `Hour` field whose values do not represent conventional hours of the day. It is not needed for daily forecasting and is excluded from the cleaned output. No derived `DateTime` helper column is created.

In [34]:
pharma = pharma.sort_values("datum").reset_index(drop=True)

date_range = pd.date_range(
    start=pharma["datum"].min(),
    end=pharma["datum"].max(),
    freq="D"
)
missing_dates = date_range.difference(pd.DatetimeIndex(pharma["datum"].dropna().unique()))

print("Chronological after sorting:", pharma["datum"].is_monotonic_increasing)
print("Expected calendar dates:", len(date_range))
print("Unique observed dates:", pharma["datum"].nunique())
print("Missing calendar dates:", len(missing_dates))
display(pd.Series(missing_dates[:20], name="first_missing_dates").to_frame())

Chronological after sorting: True
Expected calendar dates: 2106
Unique observed dates: 2106
Missing calendar dates: 0


,first_missing_dates


### 3.6 Prepare and validate the cleaned pharmaceutical dataset

In [35]:
# Keep the date, calendar descriptors, and eight sales groups.
# The Hour and DateTime helper columns are intentionally excluded.
pharma_columns = [
    "datum", "Year", "Month", "Weekday Name",
    "M01AB", "M01AE", "N02BA", "N02BE",
    "N05B", "N05C", "R03", "R06"
]
pharma_clean = pharma[pharma_columns].copy()

print("Cleaned shape:", pharma_clean.shape)
print("Missing values:")
display(pharma_clean.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", pharma_clean.duplicated().sum())
print("Duplicate dates:", pharma_clean["datum"].duplicated().sum())
print("Chronological:", pharma_clean["datum"].is_monotonic_increasing)
print("Negative sales values:")
display(pharma_clean[sales_columns].lt(0).sum().to_frame("negative_count"))

Cleaned shape: (2106, 12)
Missing values:


,missing_count
datum,0
Year,0
Month,0
Weekday Name,0
M01AB,0
M01AE,0
N02BA,0
N02BE,0
N05B,0
N05C,0


Exact duplicate rows: 0
Duplicate dates: 0
Chronological: True
Negative sales values:


,negative_count
M01AB,0
M01AE,0
N02BA,0
N02BE,0
N05B,0
N05C,0
R03,0
R06,0


In [36]:
pharma_output = os.path.join(PROCESSED_DIR, "pharma_sales_clean.csv")
pharma_clean.to_csv(pharma_output, index=False)

print("Saved:", os.path.abspath(pharma_output))
print("File created:", os.path.exists(pharma_output))

Saved: C:\Users\Administrator\MediShelf\data\processed\pharma_sales_clean.csv
File created: True


## 4. Cleaning decisions and handover

- Calendar mismatches are checked explicitly so data consistency can be reported.
- Date columns are parsed and records are sorted chronologically.
- Binary `Events` is excluded from IQR outlier detection.
- Potential outliers in numeric demand/sales fields are flagged but not automatically deleted or capped.
- `Hour` and the derived `DateTime` helper are not included in the cleaned pharmaceutical output.
- The pharmaceutical dataset's missing dates are reported rather than automatically filled with zero sales. A missing record is not necessarily evidence of zero demand.
- The blood-demand data is synthetic; unusual day-to-day population changes should be documented and considered during interpretation.